In [1]:
# Import the tuecycle package
from tuecycle.data import DataManager
from tuecycle.config import list_stations_with_weather
from tuecycle.plots import get_plot, list_plots
import pandas as pd


START_DATE = (2022, 12, 1)
END_DATE = (2025, 11, 30)

# Stations to load
STATIONS = [station for station in list_stations_with_weather()]

# Initialize the DataManager
dm = DataManager(start_date=START_DATE,
                 end_date=END_DATE)
# Load all comparison stations at once using the configuration
dfs = dm.get_multiple(STATIONS)

# Remove visual outliers
VISUAL_OUTLIERS = [
    ("freiburg_eschholz", "2025-04-06 11:00:00"),
    
    ("freiburg_wiwili", "2025-01-30 17:00:00"),
    ("freiburg_wiwili", "2025-01-30 18:00:00"),
    ("freiburg_wiwili", "2025-01-30 19:00:00"), 
    
    ("heidelberg_ploeck", "2025-04-27 08:00:00"),
    
    ("karlsruhe_erbprinzen", "2023-03-28 08:00:00"),
    ("karlsruhe_erbprinzen", "2024-06-07 19:00:00"),
    
    ("kirchheim_barometer", "2022-12-19 19:00:00"),
    
    ("loerrach_berliner", "2023-02-27 17:00:00"),
    
    ("loerrach_friedhof", "2023-05-24 21:00:00"),
    
    # ("mannheim_feudenheimstr_aus", "2024-01-27 17:00:00"),
    
    # ("mannheim_feudenheimstr_ein", "2024-01-27 15:00:00"),
    
    ("mannheim_schlosspark", "2025-06-27 09:00:00"),
    ("mannheim_schlosspark", "2024-06-12 09:00:00"),
    
    ("ravensburg_meer_auf", "2025-11-12 22:00:00"),
    ("ravensburg_meer_auf", "2023-08-20 23:00:00"),
    ("ravensburg_meer_auf", "2024-03-02 17:00:00"),
    
    # ("stuttgart_kremmler", "2024-09-15 10:00:00"),
    # ("stuttgart_kremmler", "2024-09-15 11:00:00"),
    # ("stuttgart_kremmler", "2024-09-15 12:00:00"),
    # ("stuttgart_kremmler", "2024-09-15 13:00:00"),
    
    # ("stuttgart_solitude", "2023-10-08 15:00:00"),
    # ("stuttgart_solitude", "2025-07-04 21:00:00"),
    
    ("stuttgart_waiblinger", "2023-12-21 09:00:00"),
    ("stuttgart_waiblinger", "2023-12-21 10:00:00"),
    ("stuttgart_waiblinger", "2025-09-21 16:00:00"),
    
    ("tuebingen_steinlach", "2025-03-19 14:00:00"),
]

for alias, timestamp in VISUAL_OUTLIERS:
    df = dfs[alias]
    df.loc[df["datetime"] == timestamp, 'bike'] = None
print("Visual outliers removed.\n")

# Remove days with more than ten 0 counts
for alias, df in dfs.items():
    # Count zero bike counts per day
    df['date'] = df['datetime'].dt.date
    zero_counts_per_day = df.groupby('date')['bike'].apply(lambda x: (x == 0).sum())
        
    # Identify faulty days
    faulty_days = zero_counts_per_day[zero_counts_per_day > 10].index
        
    # Remove faulty days
    for day in faulty_days:
        df.loc[df['date'] == day, 'bike'] = None
        
    # Drop the temporary 'date' column
    df.drop(columns=['date'], inplace=True)
print("Faulty days removed.\n")

# Remove Heilbronn Neckarufer outlier period (2025-10-26 00:00:00 - 2025-10-30 00:00:00)
alias = "heilbronn_neckarufer"
df = dfs[alias]
start_outlier = pd.to_datetime("2025-10-26 00:00:00")
end_outlier = pd.to_datetime("2025-10-30 00:00:00")
df.loc[(df["datetime"] >= start_outlier) & (df["datetime"] <= end_outlier), 'bike'] = None
print(f"Station {alias} - Heilbronn Neckarufer outlier period removed.\n")

# Add stations with more than 10% missing values to excluded stations
excluded_stations = []
for alias, df in dfs.items():
    missing_values = df['bike'].isna().sum()
    total_values = len(df)
    if (missing_values / total_values) > 0.1:
        excluded_stations.append(alias)
        
# Now exclude these stations from further analysis
for alias in excluded_stations:
    del dfs[alias]
print("Excluded stations with more than 10% missing values.\n")

print(f"\nData cleaning completed. Stations remaining for analysis: {len(dfs.keys())}")
STATIONS = dfs.keys()
for alias in dfs.keys():
    print(f"- {alias}")

Visual outliers removed.

Faulty days removed.

Station heilbronn_neckarufer - Heilbronn Neckarufer outlier period removed.

Excluded stations with more than 10% missing values.


Data cleaning completed. Stations remaining for analysis: 37
- freiburg_eschholz
- freiburg_gueterbahn
- freiburg_wiwili
- heidelberg_eppelheimer
- heidelberg_ploeck
- heidelberg_theodor_heuss
- heilbronn_neckarufer
- heilbronn_nord
- karlsruhe_erbprinzen
- kirchheim_barometer
- konstanz_herose
- loerrach_berliner
- loerrach_friedhof
- ludwigsburg_alleen
- ludwigsburg_favorite
- mannheim_konrad_adenauer
- mannheim_kurpfalz
- mannheim_luzenberg
- mannheim_renz
- mannheim_schlosspark
- mannheim_schwetzinger
- ravensburg_bahnhof
- ravensburg_eishalle
- ravensburg_meer_ab
- ravensburg_meer_auf
- stuttgart_insel
- stuttgart_koenig_karls
- stuttgart_kraeherwald
- stuttgart_lautenschlager
- stuttgart_samara
- stuttgart_stuttgarter
- stuttgart_taubenheim
- stuttgart_waiblinger
- stuttgart_tuebinger
- tuebingen_tunnel

In [2]:
import numpy as np

# Combine raw bike counts per city, then standardize once at city level
cities = set(alias.split('_')[0] for alias in STATIONS)
city_dfs = {}
for city in cities:
    city_dfs[city] = pd.DataFrame()
    city_dfs[city]['datetime'] = dfs[next(iter(dfs))]['datetime']
    
    # Average raw bike counts
    bike_counts = pd.concat(
        [dfs[alias]['bike'] for alias in STATIONS if alias.startswith(city)],
        axis=1
    )
    # and only keep rows where ALL counters have values
    city_dfs[city]['bike_count'] = bike_counts.mean(axis=1, skipna=False)
    
    # Standardize at city level to remove position bias and keep relative changes
    city_dfs[city]['standardized_bike_count'] = (
        (city_dfs[city]['bike_count'] - city_dfs[city]['bike_count'].mean()) / 
        city_dfs[city]['bike_count'].std()
    )
    
    # Add logged bike counts
    city_dfs[city]['logged_bike_count'] = np.log1p(city_dfs[city]['bike_count'])
    
    # Also average weather data
    city_dfs[city]['temp'] = pd.concat(
        [dfs[alias]['temp'] for alias in STATIONS if alias.startswith(city)],
        axis=1
    ).mean(axis=1, skipna=False)
    city_dfs[city]['rain'] = pd.concat(
        [dfs[alias]['rain'] for alias in STATIONS if alias.startswith(city)],
        axis=1
    ).mean(axis=1, skipna=False)

# For each city print the amount of missing values in bike_count
for city in cities:
    missing_count = city_dfs[city]['bike_count'].isna().sum()
    total_count = len(city_dfs[city])
    print(f"{city.capitalize()} – missing bike_count values: {missing_count} out of {total_count} ({(missing_count/total_count)*100:.2f}%)")
    

Heidelberg – missing bike_count values: 2292 out of 26304 (8.71%)
Kirchheim – missing bike_count values: 77 out of 26304 (0.29%)
Heilbronn – missing bike_count values: 1671 out of 26304 (6.35%)
Freiburg – missing bike_count values: 802 out of 26304 (3.05%)
Stuttgart – missing bike_count values: 514 out of 26304 (1.95%)
Tuebingen – missing bike_count values: 1860 out of 26304 (7.07%)
Ulm – missing bike_count values: 814 out of 26304 (3.09%)
Konstanz – missing bike_count values: 220 out of 26304 (0.84%)
Ludwigsburg – missing bike_count values: 76 out of 26304 (0.29%)
Mannheim – missing bike_count values: 1863 out of 26304 (7.08%)
Ravensburg – missing bike_count values: 2770 out of 26304 (10.53%)
Loerrach – missing bike_count values: 234 out of 26304 (0.89%)
Karlsruhe – missing bike_count values: 275 out of 26304 (1.05%)


In [3]:
# Public holidays in baden-wuerttemberg
public_holidays = [
    "2022-12-25",
    "2022-12-26",
    "2023-01-01",
    "2023-01-06",
    "2023-04-07",
    "2023-04-10",
    "2023-05-01",
    "2023-05-18",
    "2023-05-29",
    "2023-06-08",
    "2023-10-03",
    "2023-11-01",
    "2023-12-25",
    "2023-12-26",
    "2024-01-01",
    "2024-01-06",
    "2024-03-29",
    "2024-04-01",
    "2024-05-01",
    "2024-05-09",
    "2024-05-20",
    "2024-05-30",
    "2024-10-03",
    "2024-11-01",
    "2024-12-25",
    "2024-12-26",
    "2025-01-01",
    "2025-01-06",
    "2025-04-18",
    "2025-04-21",
    "2025-05-01",
    "2025-05-29",
    "2025-06-09",
    "2025-06-19",
    "2025-10-03",
    "2025-11-01"
]

# Add day_of_week, is_public_holiday, and is_workday feature to each city's dataframe
for city in cities:
    city_dfs[city]['date'] = city_dfs[city]['datetime'].dt.date
    city_dfs[city]['day_of_week'] = city_dfs[city]['datetime'].dt.dayofweek  # Monday=0, Sunday=6
    city_dfs[city]['is_public_holiday'] = city_dfs[city]['date'].astype(str).isin(public_holidays).astype(int)
    
    # Workday and Non-Workday indicators
    city_dfs[city]['is_workday'] = (
        (city_dfs[city]['day_of_week'] < 5) &   # Monday to Friday
        (~city_dfs[city]['is_public_holiday'])  # Not a public holiday
    ).astype(int)
    city_dfs[city]['is_non_workday'] = (
        (city_dfs[city]['is_workday'] == 0)
    ).astype(int)
    
    # Leisure hour indicator (Saturday and Sunday from 10am to 6pm)
    city_dfs[city]['is_leisure'] = (
        ((city_dfs[city]['day_of_week'] == 5) | (city_dfs[city]['day_of_week'] == 6)) &  # Saturday or Sunday
        (city_dfs[city]['datetime'].dt.hour >= 10) & (city_dfs[city]['datetime'].dt.hour <= 17)
    ).astype(int)
    
    # Add hour, month, year features
    city_dfs[city]['hour'] = city_dfs[city]['datetime'].dt.hour
    city_dfs[city]['month'] = city_dfs[city]['datetime'].dt.month
    city_dfs[city]['year'] = city_dfs[city]['datetime'].dt.year
    
    # Drop temporary column
    city_dfs[city].drop(columns=['date'], inplace=True)


In [4]:
import numpy as np
import pandas as pd

def _empirical_cdf(values, reference):
    """Return empirical CDF ranks in [0,1] for `values` based on `reference`."""
    ref = pd.Series(reference).dropna().to_numpy()
    if ref.size == 0:
        return pd.Series(np.nan, index=values.index)
    ref = np.sort(ref)
    vals = values.to_numpy()
    ranks = np.searchsorted(ref, vals, side='right') / ref.size
    return pd.Series(ranks, index=values.index)

def add_bad_weather_index(
    df,
    temp_col='temp',
    rain_col='rain',
    reference_df=None,
    weights=(0.5, 0.5),
    output_col='bad_weather_index'
    ):
    """
    Add a data-driven bad weather index using empirical CDFs.
    - No fixed temperature or rain ranges are assumed.
    - Uses `reference_df` to define the empirical distributions (defaults to df).
    - Colder temperatures => higher index; more rain => higher index.
    """
    ref = df if reference_df is None else reference_df
    temp_ref = ref[temp_col]
    rain_ref = ref[rain_col]
    
    temp_cdf = _empirical_cdf(df[temp_col], temp_ref)
    rain_cdf = _empirical_cdf(df[rain_col], rain_ref)
    
    # Temperature: lower is worse => invert CDF
    temp_index = 1 - temp_cdf
    rain_index = rain_cdf
    
    w_temp, w_rain = weights
    w_sum = w_temp + w_rain
    if w_sum == 0:
        raise ValueError("weights must not sum to zero")
    w_temp, w_rain = w_temp / w_sum, w_rain / w_sum
    
    df = df.copy()
    df[output_col] = (w_temp * temp_index) + (w_rain * rain_index)
    return df

In [5]:
# Apply bad weather index per city (city-calibrated)
for city in cities:
    city_dfs[city] = add_bad_weather_index(
        city_dfs[city],
        temp_col='temp',
        rain_col='rain',
        reference_df=city_dfs[city],  # Use city's own distribution as reference
        weights=(0.5, 0.5),
        output_col='bad_weather_index'
    )
print("Bad weather index added to each city's dataframe.")

Bad weather index added to each city's dataframe.


In [6]:
# Log-linear regression 
import statsmodels.api as sm
results = {}

leisure_dfs = {}
for city in cities:
    leisure_dfs[city] = city_dfs[city][city_dfs[city]['is_leisure'] == 1]

for city in cities:
    df = leisure_dfs[city].dropna(subset=['logged_bike_count', 'rain', 'temp'])

    # Build X with rain, temp, and hour dummies
    X = df[['rain', 'temp']].copy()
    X = X.astype(float)
    
    # Fixed Effects: month, year, hour
    fe_hour  = pd.get_dummies(df['hour'],  prefix='hour',  drop_first=True, dtype=float)
    fe_month = pd.get_dummies(df['month'], prefix='month', drop_first=True, dtype=float)
    
    # Combine all regressors
    X = pd.concat([X, fe_month, fe_hour], axis=1)
    
    X = sm.add_constant(X)  # Adds a constant term to the predictor
    y = df['logged_bike_count'].astype(float)
    
    model = sm.OLS(y, X).fit()
    results[city] = model
    print(f"\nLog-linear regression results for {city.capitalize()}:")
    print(model.summary())


Log-linear regression results for Heidelberg:
                            OLS Regression Results                            
Dep. Variable:      logged_bike_count   R-squared:                       0.299
Model:                            OLS   Adj. R-squared:                  0.293
Method:                 Least Squares   F-statistic:                     48.55
Date:                Thu, 29 Jan 2026   Prob (F-statistic):          4.41e-159
Time:                        02:09:40   Log-Likelihood:                -868.37
No. Observations:                2296   AIC:                             1779.
Df Residuals:                    2275   BIC:                             1899.
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const

In [7]:
# Coefficients extraction
for city in cities:
    model = results[city]
    coef = model.params
    print(f"\nCity: {city.capitalize()}")
    print(f"Intercept: {coef['const']:.4f}")
    print(f"Rain coefficient: {coef['rain']:.4f}")
    print(f"Temperature coefficient: {coef['temp']:.4f}")

# Add interpretation calculations
for city in cities:
    model = results[city]
    coef = model.params
    
    rain_coef = coef['rain']
    temp_coef = coef['temp']
    
    rain_effect_1mm = (np.expm1(rain_coef * 1)) * 100  # Effect of 1 mm increase in rain
    temp_effect_5C = (np.expm1(temp_coef * 5)) * 100   # Effect of 5 °C increase in temperature
    
    print(f"\nCity: {city.capitalize()}")
    print(f"Effect of 1 mm increase in rain: {rain_effect_1mm:.2f}% change in bike counts")
    print(f"Effect of 5 °C increase in temperature: {temp_effect_5C:.2f}% change in bike counts")


City: Heidelberg
Intercept: 4.8715
Rain coefficient: -0.2356
Temperature coefficient: 0.0136

City: Kirchheim
Intercept: 3.2830
Rain coefficient: -0.2757
Temperature coefficient: 0.0494

City: Heilbronn
Intercept: 3.0322
Rain coefficient: -0.2423
Temperature coefficient: 0.0576

City: Freiburg
Intercept: 4.9105
Rain coefficient: -0.1374
Temperature coefficient: 0.0168

City: Stuttgart
Intercept: 3.1220
Rain coefficient: -0.2428
Temperature coefficient: 0.0535

City: Tuebingen
Intercept: 5.1247
Rain coefficient: -0.1235
Temperature coefficient: 0.0314

City: Ulm
Intercept: 2.8612
Rain coefficient: -0.2498
Temperature coefficient: 0.0782

City: Konstanz
Intercept: 5.3108
Rain coefficient: -0.2532
Temperature coefficient: 0.0376

City: Ludwigsburg
Intercept: 3.7497
Rain coefficient: -0.2871
Temperature coefficient: 0.0441

City: Mannheim
Intercept: 3.8961
Rain coefficient: -0.2242
Temperature coefficient: 0.0296

City: Ravensburg
Intercept: 2.1854
Rain coefficient: -0.2365
Temperature co

In [8]:
# Rank cities by effects
rain_effects_per_mm = {}
temp_effects_per_5C = {}
rain_ci_per_mm = {}
temp_ci_per_5C = {}
for city in cities:
    model = results[city]
    coef = model.params
    rain_coef = coef['rain']
    temp_coef = coef['temp']
    rain_se = model.bse['rain']
    temp_se = model.bse['temp']
    rain_effect_1mm = (np.expm1(rain_coef * 1)) * 100  # Effect of 1 mm increase in rain
    temp_effect_5C = (np.expm1(temp_coef * 5)) * 100   # Effect of 5 °C increase in temperature
    rain_effects_per_mm[city] = rain_effect_1mm
    temp_effects_per_5C[city] = temp_effect_5C
    
    # Calculate confidence intervals in effect space
    # Rain
    lower_rain_bound = rain_coef - 1.96 * rain_se
    upper_rain_bound = rain_coef + 1.96 * rain_se
    lower_rain_effect = (np.expm1(lower_rain_bound * 1)) * 100
    upper_rain_effect = (np.expm1(upper_rain_bound * 1)) * 100
    rain_ci_per_mm[city] = (lower_rain_effect, upper_rain_effect)
    
    # Temperature
    lower_temp_bound = temp_coef - 1.96 * temp_se
    upper_temp_bound = temp_coef + 1.96 * temp_se
    lower_temp_effect = (np.expm1(lower_temp_bound * 5)) * 100
    upper_temp_effect = (np.expm1(upper_temp_bound * 5)) * 100
    temp_ci_per_5C[city] = (lower_temp_effect, upper_temp_effect)

# Display rankings
print("\nCities ranked by rain effect (1 mm increase):")
for city, effect in sorted(rain_effects_per_mm.items(), key=lambda item: item[1], reverse=True):
    ci = rain_ci_per_mm[city]
    print(f"- {city.capitalize()}: {effect:.2f}% (95% CI: {ci[0]:.2f}%, {ci[1]:.2f}%)")
print("\nCities ranked by temperature effect (5 °C increase):")
for city, effect in sorted(temp_effects_per_5C.items(), key=lambda item: item[1], reverse=True):
    ci = temp_ci_per_5C[city]
    print(f"- {city.capitalize()}: {effect:.2f}% (95% CI: {ci[0]:.2f}%, {ci[1]:.2f}%)")


Cities ranked by rain effect (1 mm increase):
- Tuebingen: -11.62% (95% CI: -14.29%, -8.87%)
- Freiburg: -12.84% (95% CI: -14.97%, -10.65%)
- Loerrach: -16.18% (95% CI: -18.42%, -13.89%)
- Karlsruhe: -18.58% (95% CI: -22.26%, -14.72%)
- Mannheim: -20.08% (95% CI: -22.59%, -17.50%)
- Heidelberg: -20.99% (95% CI: -24.04%, -17.81%)
- Ravensburg: -21.06% (95% CI: -24.13%, -17.87%)
- Heilbronn: -21.51% (95% CI: -24.34%, -18.58%)
- Stuttgart: -21.56% (95% CI: -24.16%, -18.87%)
- Ulm: -22.10% (95% CI: -25.30%, -18.77%)
- Konstanz: -22.37% (95% CI: -24.85%, -19.81%)
- Kirchheim: -24.09% (95% CI: -28.31%, -19.63%)
- Ludwigsburg: -24.95% (95% CI: -28.56%, -21.16%)

Cities ranked by temperature effect (5 °C increase):
- Ulm: 47.82% (95% CI: 44.51%, 51.20%)
- Heilbronn: 33.34% (95% CI: 30.67%, 36.08%)
- Stuttgart: 30.67% (95% CI: 28.60%, 32.78%)
- Ravensburg: 30.06% (95% CI: 27.09%, 33.09%)
- Kirchheim: 28.04% (95% CI: 24.52%, 31.66%)
- Ludwigsburg: 24.69% (95% CI: 22.21%, 27.22%)
- Loerrach: 22.